# TEMA 5: PUESTA EN PRODUCCIÓN Y CICLO DE VIDA DE LOS MODELOS - PARTE 1

La idea principal es entender que el prototipo = entrenar un modelo es relativamente fácil. Sin embargo, llevarlo a producción desplegando el modelo no es tan fácil

> ERROR COMÚN: pensar que entrenar y validar un modelo guardándolo en un .pkl y un .joblib es suficiente

Para que esté en producción el modelo debe ser usado y DESPLEGADO. Serializándolo no es suficiente, debe responder a las preguntas reales. 

## ¿Qué significa que un modelo esté desplegado?

DESPLEGAR (DEPLOY) un modelo consiste en convertir el modelo en un servicio accesible 

## ¿Cuándo va a hacer predicciones mi modelo= consumir mi modelo?

En base a la respuesta a esta pregunta entran en juego dos paradigmas:

### BATCH SCORING

Batch significa lote. EL modelo no trabaja continuamente, se ejecuta en una hora determinada y calcula muchas predicciones de golpe. Por tanto, procesa un lote completo.

> EJEMPLO: empresa que ejecuta su modelo cada dia a las 2 a.m y guarda las predicciones de riesgo o de demanda de productos en un fichero. Las predicicones se hacen de forma diferida y no está a la espera de solicitudes. 


+ Claramente es más barato ya que no necesitas tener el servidor encendido escuchando. Solo enciendes cuando toca procesar el lote. 
+ Además es más simple, a priori no se necesitas una respuesta constante -> menos requisitos de infraestructura. 
+ Más fácil de controlar porque literalmente puedes limpiar y validar los datos antes de ejecutar el modelo y hacer inferencia, de maenra que todo se controla de manera mucho más global

### ONLINE SERVING

¿Qué pasa si necesitas respuestas inmediatas? En este caso, entra el online serving en juego que expone el modelo como un servicio accesible mediante una API, de manera que cada petición genera una predicción al instante. Esto mismo es lo que permite integrar el modelo en flujos, sistemas en tiempo real o cualquier aplicación que requiera sistemas en tiempo real. 

- Claramente se le suma una complejidad técnica que antes no teníamos. Al exponer el modelo como un servicio accesible en tiempo real necesitamos desplegar servidores de manera que se mantengan los modelos en memoria, la latencia sea mínima, se pueda escalar horizontalmente cuando el tráfico aumenta, registrar predicciones y recuperarse ante fallos. 
- Además, el modelo debe ser robusto ante entradas imprevistas y degradación silenciosa
- TODO ESTO LO HACE SER MÁS COSTOSO, Y MÁS DÍFICIL DE MANTENER. 

### ¿Cómo elijo una para mi caso de uso?

Hay que evaluar multiples dimensiones:

- LATENCIA: cuánto tiempo podemos permitirnos esperar para una predicción (en batch importa poca pero en online serving es muy importante)
- FRECUENCIA: cada cuánto habrán que generar predicciones (en batch la frecuencia es más baja que online)
- VOLUMEN: cuantas predicciones por unidad de tiempo habrán (en batch muchas predicciones juntas, volumen grande- en online una cada vez)
- CONTEXTO TÉCNICO: capacidad para mantener servicio activo y escalable si hay muchas predicciones o no
- TRAZABILIDAD: necesitamos registrar cada petición individual o no (en batch tienes un fichero con todo-> es más facil de mantener trazabilidad)

| Característica      | Batch                  | Online            |
| ------------------- | ---------------------- | ----------------- |
| Momento             | Programado             | Inmediato         |
| Latencia            | No crítica             | Muy importante    |
| Coste               | Bajo                   | Más alto          |
| Complejidad         | Baja                   | Alta              |
| API                 | No necesaria           | Sí                |
| Disponibilidad 24/7 | No                     | Sí                |
| Ejemplo             | Riesgo diario clientes | Fraude en tarjeta |

> Se pueden tener enfoques híbridos-> predicción base en batch y refinas en online cuando pide un préstamo. Equilibras el coste, rendimiento y precisión.


## DESPLIGUE COMO API REST CON Flask/FastAPI

Imagina que después de analizar todos los puntos clave para elegir entre Batch Scoring y Online Serving, es decir, latencia, frecuencia, volumen, contexto técnico y trazabilidad, se decide que el modelo va a ser expuesto como un servicio. 

Se construye una API REST, es decir, un punto de acceso web que recibe solicitudes HTTP. Procesa datos de entrada, ejecuta el modelo y devuelve una predicción en formato estructurado. 

Sistema externo (de donde viene la solicitud:cliente)
      ↓
   API REST (punto de acceso web que permite que se exponga el modelo con endpoints)
      ↓
   Modelo ML
      ↓
 Predicción estructurada en formato JSON de vuelta

> Desplegar un modelo como API REST significa exponerlo mediante un endpoint HTTP, normalmente /predict, para que otros sistemas puedan enviar datos de entrada y recibir una predicción estructurada, habitualmente en formato JSON.

De hecho permite que el modelo se pueda utilizar en distintas arquitecturas modernas como aplicación web, CRM, POwer BI. Y todas con el endpoint /predict y la solicitud HTTP pueden comunicarse con el modelo sin conocer el código interno del modelo ni nada. SE VUELVE UN SERVICIO REUTILIZABLE (YA NO VIVE EN NOTEBOOK) QUE SE PUEDE USAR Y SER CONSUMIDO SIN DUPLICAR EL MODELO.


### Flask

Sabemos que para poder llevar a cabo Online Batching necesitamos construir una API REST como punto de acceso web y que el modelo pueda ser consumido gracias al endpoint de la API con peticiones HTTP. Hay dos herramientas en Python con la que construirla. 


**from flask import Flask**

+ Flask fue el estándar durante muchos años. Es simple y fácil de aprender, ligera y flexible y por eso su popularidad. 

- Lo negativo es super claro. Muchas cosas tienes que configurarlas tú mismo, por ejemplo la documentación, validación de los datos que llegan por la solicitud HTTP  a la API REST, optimizaciones. Todoe sto no viene tan automatizado.


### FastAPI

Es más moderno y se piensa para las APIs de alto rendimiento. Por tanto, es más rápido y tiene buen rendimiento

**from fastapi import FastAPI**

+ Soluciona lo negativo de Flask. Por un lado hay documentación automática OpenAPI, en /docs puedes probar la api sin poner nada por terminal. Además, hay validación de datos y detecta los errores que se necesiten . Por último, hay optimizaciones para escribir mucho menos código-> tareas ya incorporadas.

+ ASINCRONÍA, FastAPI gestiona mejor muchas peticiones HTTP simultáneas a la API REST QUE SE CREA sin bloquearse. Por tanto, suele escalar mejor en esos casos. 

| Flask                | FastAPI               |
| -------------------- | --------------------- |
| Más veterano         | Más moderno           |
| Muy flexible         | Muy productivo        |
| Fácil de aprender    | Muy usado en ML       |
| Menos automatización | Validación automática |
| Menos rendimiento    | Mayor rendimiento     |


## ¿Cómo encaja MLOps en todo esto? 

Una vez vemos el sentido de como elegir que paradigma se necesita en la práctica para nuestro modelo, y además como se puede tener el modelo como un servicio, es necesario encajar que es MLOps en todo este proceso. Esto se debe a que es lo que da forma y orden a todos estos pasos

**MLOps = Machine Learning + DevOps + Ingeniería de datos.**

Es el conjunto de prácticas, herramientas y cultura que permiten llevar modelos de ML a producción de forma automatizada, reproducible y mantenible a lo largo del tiempo. 

Gracias a MLOps podemos dar vida a estos modelos más alla de un notebook.

### EMPAQUETADO/SERIALIZACIÓN 

EL proceso típico de despliegue, una vez se tiene entrenado el modelo, zomienza persistiendo/encapsulando el modelo, habitualmente se serializa con joblib y pickle (muy simple y directo). ESTE ENCAPSULAMIENTO DEL MODELO INCLUYE LOS PESOS/PARÁMETROS. 

**joblib.dump(modelo, "modelo_rf.pkl")**

### DESPLIEGUE

Evidentemente llegados al punto de tener el modelo serializado o encapsulado o empaquetado, tenemos que desplegar el modelo, SIEMPRE CON LA CABEZA EL BATCH SCORING O ONLINE SERVING YA QUE CAMBIA TODO AL SER DISTINTOS PARADIGMAS. Recordamos que supusimos que al usar Online Serving creábamos una API REST que permite recibir solicitudes HTTP y dar una respuesta del modelo, ya que este queda expuesto como un servicio en el endpoint de la api. Hasta aquí todo lo hemos visto. Seguimos con el paradigma de Online Serving y la API REST creada:

#### ¿Qué ocurre cuando alguien llama a /predict endpoint de la API REST creada con Flask o FastAPI?

Cuando arranca la API donde está expuesto el modelo, el modelo se carga en memoria. 

**modelo = joblib.load("modelo_rf.pkl")** -> joblib es una libreria de Python que sirve para cargar y guardar modelos

Esto lo que hace es leer el archivo serializado donde está el modelo y reconstruye el modelo en memoria al arrancar la API. 

> NÓTESE QUE NO SE CARGA AL REALIZAR CADA PREDICCIÓN SINO AL ARRANCAR LA API, ESTO SE DEBE A QUE SI SE CARGA EN CADA PREDICCIÓN SERÍA MUY MUY LENTO. EL MODELO SE CARGA UNA VEZ AL ARRANCAR LA API, SE QUEDA EN MEMORIA AL ARRANCAR Y RECIBE LAS MILES DE PETICIONES (NO SE CARGA MIL VECES)

Si llega una petición HTTP a la API REST(arrancada con el modelo en memoria) recibe los datos, que viajan por internet hasta la API. LLegan en formato JSON que no es mas que el formato para intercambiar datos con la API. En la API el modelo está expuesto con un endpoint /predict, porque envía información. Por tanto el endpoint recibe mediante una petición POST

> SI SOLO OBTUVIESE INFORMACIÓN EL ENDPOINT SIN ENVÍAR INFORMACIÓN RECIBE PETICIONES GET, NO PETICIONES POST COMO EL /PREDICT EN ML

Llegamos al punto donde la API recibe la solicitud POST, es decir, los datos en formato JSON. FastAPI recibe el JSON y lo convierte en un objeto Python (Pydantic)

> Pydantic VALIDA LOS DATOS, permite especificar en FastAPI el tipo de datos que esperas para poder asi detectar los fallos

Nótese que una vez lo tienes en objeto Python puede necesitar escalado y de hecho EL MISMO PREPROCESAMIENTO QUE TUVO EL MODELO AL ENTRENAR-> Sino falla gravemente. Esto ocurre cuando quieres predecir desplegando o sin desplegar

Finalmente, con los datos habiendo llegado y con las transformaciones y validaciones que se necesiten, se realiza la predicción gracias al modelo, recibiendo una respuesta que debe ser enviada de vuelta.

La respuesta es clara, la API devuelve otro JSON (petición POST)

#### Terminología interesante

- Serializar el modelo con joblib
- Exponer el modelo como un servicio en tiempo real
- API REST punto de acceso web al modelo que recibe peticiones HTTP
- Modelo expuesto con un endpoint /predict dentro de la API
- API creada con Flask o FastAPI
- Carga en memoria del modelo al arrancar la API y no al realizar cada petición de manera individual
- Pydantic para validar el JSON de entrada y transformarlo en objeto Python validado
- Petición POST al endpoint /predict para enviar las variables de entrada del modelo dentro del cuerpo de la solicitud y obtener una predicción como respuesta


#### Una API en local no es una API de producción 

Se utiliza uvicorn main:app --reload para ejecutar FastAPI en tu ordenador en local. Uvicorn no es más que el servidor que ejecuta FastAPI en local. De hecho significa:

main.py
↓
ejecuta objeto app
↓
lanza servidor web

RECORDAR ES EN LOCAL-> FUNCIONA EN MI ORDENADOR, ES PARA PRUEBAS NO PARA QUE FUNCIONE EN TODA LA EMPRESA. Uvicorn es el servidor que arranca y ejecuta la aplicación FastAPI, abriendo un puerto por donde escuchar.

> De hecho, un modelo funcional no implica que sea fiable. ¿Qué pasa si falla? ¿Dónde queda registrado? LOGGING ¿Qué ocurre si alguien envía basura? DOCUMENTACIÓN PARA QUE SEPAN USAR TU API ¿Qué ocurre si atacan la API? AUTENTIFICACION ¿Qué ocurre si actualizas el modelo? VERSIONADO DE MODELOS


#### CONCEPTOS CLAVE QUE MENCIONA EL TEMARIO EN ESTE PUNTO 

- HTTP: Es el **protocolo** con el que se comunican las aplicaciones web. Cuando alguien llama a: POST /predict (Petición de tipo POST), lo que realmente está enviando es una petición HTTP (petición que sigue ese protocolo para comunicarse con la API EL CLIENTE).

- Status Codes: Son códigos que indican qué ha pasado.

> 200 Todo correcto. 400 Error del cliente (ENVIA DATOS QUE NO). 401 No autenticado. 403 Prohibido. 500 Error interno.

- Headers: Son metadatos de la petición. Content-Type: application/json-> Lo que te envío es JSON.

- **JWT** JSON Web Token-> Es una forma de autenticar usuarios.

> Sin JWT Cualquiera podría llamar a: POST /predict
> Con JWT. El usuario primero se identifica. Recibe un token. Después envía: Authorization: Bearer eyJhbGciOi... La API verifica el token. Si es válido: Acceso permitido. Si no: 401 Unauthorized

- Training-Serving Skew: Diferencia entre cómo se generan las variables durante el entrenamiento y cómo se generan durante el servicio en producción

- Una Feature Store es un lugar centralizado donde se definen y almacenan las variables (features).La misma lógica se reutiliza. No hay dos implementaciones distintas. No hay riesgo de inconsistencias. Permite **lineage**-> de donde salió la variable hay un versionado de variables


#### Docker: idea 

Seguimos con el despliegue del modelo. Hemos visto que en esta fase en un proyecto de ML con MLOps había que exponer el modelo como un servicio listo para ser consumido gracias a la creacion de una API REST que sigue el protocolo HTTP, es decir, recibir peticiones HTTP del tipo que sean. 

Bajo esta premisa, y teniendo un modelo expuesto en un endpoint y además una API que funciona correctamente-> puede ocurrir que al enviarlo a otro servidor falle por completo por problemas de versiones y dependencias locales (ejecutar en una máquina distinta). Por eso aparece DOCKER, para evitar que funcione en tu ordenador y que en producción no funcione.

CON DOCKER NO SOLO ENVÍO EL CÓDIGO SINO QUE ENVÍO TODO EL ENTORNO. En lugar de enviar: main.py y modelo.pkl, envías:

- Código
+
- Modelo
+
- Python
+
- Librerías
+
- Configuración -> todo empaquetado junto.


> Build Once, Run Anywhere FILOSOFÍA DE MLOps.

> Docker permite encapsular el modelo, el código y todas sus dependencias en una imagen reproducible, garantizando que el comportamiento sea idéntico entre desarrollo, pruebas y producción.

#### Conceptos clave de Docker

- Imagen (Image). Es el paquete completo. 
Piensa en ella como una fotografía congelada del entorno.

docker build -t predictor-vuelos . 

- Contenedor (Container).Es una instancia ejecutándose de una imagen. Ejemplo en el CMD, pongo que en el contenedor este escuchando uvicorn en el 8000. Además al runnear un contenedor estoy ejecutando lo que pone en el CMD y a su vez creando una conexión host:contenedor

docker run -p 8000:8000 predictor-vuelos

- Dockerfile Porque Docker por sí solo no sabe qué empaquetar en la imagen. Necesita una especie de receta llamada Dockerfile que le diga:

Usa Python 3.10
↓
Instala estas dependencias
↓
Copia mi código
↓
Arranca FastAPI con Uvicorn

#### FLUJO REAL DE LLEGADA DE PETICION

Cliente
↓
Puerto 8000 del host
↓
Docker
↓
Puerto 8000 del contenedor
↓
Uvicorn escuchando alli-> en el Dockerfile se pone que en el 8000 del contenedor este uvicorn y al hacer run del contenedor se ejecuta el CMD-> arranca api modelo en memoria y ademas se queda alli escuchando, con su conexion al 8000 del host (en el run pones los -p)
↓
FastAPI 
↓
Modelo
↓
Respuesta

#### Versionado de imagenes 

Gracias a las imagenes puedes hacer ROLLBACK, básicamente volver a una versión anterior estable. Tu puedes tener varias imagenes, cada una con su mundo y sus cosas de un modelo concreto. Además esa imagen tras ser desplegada puede tener problemas y querer volver a otra versión anterior. Gracias a las imagenes que tienes guardadas es muy sencillo hacer **rollback**.

> recordatorio usa -t tags al crear la imagen para poder tener las distintas versiones y tener reproducibilidad de los distintso despliegues

#### DOCKER HUB Y REPOSITORIOS PARA REGISTROS

Igual que GitHub almacena código. Docker tiene repositorios de imágenes.

- Docker Hub
- Amazon ECR
- Google Artifact Registry
- Azure Container Registry

#### docker-compose

Se puede tener una especie de docker.compose.yml para describir todos los contenedores y levantarlos a la vez. No solo vas a tener un contenedor para la API, tambien puedes tener uno para PostgreSQL etc... 

> Muy habitual: API FastAPI con el Modelo más Base de datos más Sistema de logging Todo gestionado por Compose.

**docker compose up**-> TODO ARRANCA JUNTO 


#### OPERATIVO EN DOCKER-> para disminuir latencia y coste

- Imágenes ligeras (FROM CON SLIM)
- Capas cacheables: docker construye la imagen por capas (por eso el doble copy con requirements, evitando que se instalen dependencias de nuevo si solo cambias código)
- Variables de entorno para no tener que cambiar cosas que luego te quiten tiempo ( variables de entorno para lo que cambia entre entornos)

### Plataformas de Cloud para despliegue gestionado

Aunque es técnicamente posible desplegar un modelo de machine learning de forma manual en un servidor local, lo cierto
es que este enfoque presenta limitaciones importantes cuando queremos escalar, automatizar y garantizar alta disponibilidad. En
escenarios reales, especialmente en empresas con infraestructuras distribuidas o servicios expuestos a clientes, lo habitual es optar
por plataformas de despliegue gestionado en la nube.

Estas plataformas permiten abstraer muchos de los aspectos
más complejos del despliegue: aprovisionamiento de máquinas,
balanceo de carga, gestión de certificados, escalado automático,
monitorización o integración con otros servicios de datos. En lugar
de centrarnos en configurar instancias y redes, nos enfocamos en
definir qué queremos desplegar, con qué parámetros, y en qué
condiciones debe escalar o reiniciarse. Esto permite a los equipos
de machine learning reducir la carga operativa y concentrarse en el
modelo y sus métricas.

> Las plataformas cloud dicen: Tú sube la imagen Docker y yo me encargo de la infraestructura.

1. Google Cloud Run->  Basado en Docker. Subes imagen Docker. Escalado automático. Puede llegar a: 0 instancias si nadie lo usa. Pagas por uso.

2. AWS SageMaker. Especializado en Machine Learning. Además de desplegar: Modelos, Versiones, Monitorización, Pipelines ML

3. Azure ML Endpoints. Muy parecido. Todo integrado en Azure.

Todas estas plataformas permiten subir una imagen previamente
construida (como vimos en el apartado anterior), definir recursos
(CPU, memoria), configurar variables de entorno, habilitar
autenticación y establecer políticas de escalado. Por ejemplo,
un equipo que ha desarrollado un modelo para predecir roturas
de stock en una cadena de supermercados puede contenerizar
la solución y desplegarla en Google Cloud Run en menos de 15
minutos, recibiendo predicciones a través de HTTPS sin necesidad de
configurar manualmente un servidor o firewall. Si la demanda crece
durante el Black Friday, la plataforma escala automáticamente; si no
hay tráfico, el servicio se suspende sin costes adicionales.

VENTAJAS: ESCALADO AUTOMATICO, SI FALLA, SEGURIDAD Y MONITORIZACIÓN

DESVENTAJAS: DEPENDENCIA DEL PROVEEDOR CLOUD- Las plataformas
gestionadas implican una cierta dependencia del proveedor, y
pueden ser más caras que alternativas autoalojadas cuando el
volumen de tráfico es muy alto y constante. 

> Vendor Lock-In significa que te vuelves dependiente de un proveedor tecnológico concreto y que cambiarte a otro proveedor en el futuro resulta costoso, complejo o requiere mucho trabajo.Por ejemplo, imagina que despliegas todo tu proyecto en: Amazon Web Services. Todo funciona perfectamente. Pero dentro de dos años decides irte a: Microsoft Azure y descubres que: Las APIs son distintas. Los servicios tienen nombres diferentes. Los SDKs cambian.Entonces tienes que reescribir parte de la infraestructura.

> Conviene además destacar que estas plataformas están diseñadas para integrarse con otros servicios cloud: bases de datos,-> arquitecturas completas y modulares en torno al modelo desplegado, manteniendo una visión unificada de toda la infraestructura desde una sola plataforma

## DESPLIEGUE PROGRESIVO

Conocemos todo lo importante para desplegar un modelo siguiente MLOps. Hemos pasado desde paradigmas de como consumir el modelo, de API REST y peticiones HTTP en local, librerias para crear la API, Pydantic y endpoints. Papel que desempeña Uvicorn en todo esto y conceptos clave para entender como JTW, HTTP, Status Code, Headers, Feature Store. Una vez controlabamos como la idea integra de usar una API REST para exponer el modelo en un endpoint y todo lo que eso conlleva, hablamos de imágenes en Docker, como crearlas, el Dockerfile, su significado, arrancar contenedores, ventajas y la idea principal por la que se usa. Entendimos las diferencias estructurales de no usar un contenedor a nivel de puerto y si usarlo localmente. Una vez conocímos Docker, vimos docker-compose entre otras cosas. Finalmente, vimos como distribuir las imágenes en Registry (ejemplo Docker Hub). Finalmente vimos como utilizar instanciar un servidor en la nube (dificultades para el escalado y hacerlo manual en local) y distintos ejemplos de plataformas cloud para el despliegue gestionado (te despreocupas de la parte más manual y te encargas del modelo, la imagen y al subirla al servidor con docker pull comienzan a verse las ventajas). CERRAMOS DESPLIEGUE

Tengo un modelo v1 en producción. Lo tengo que sustituir por un modelo v2. No es una buena idea APAGAR UNO Y ENCENDER EL OTRO(big bang dev o algo así). PORQUE PUEDE FALLAR V2 O no actuar como nos esperábamos. Para ello, empleo tres patrones:

1. BLUE-GREEN DEPLOY (se usa cuando tienes bastante claro que va a funcionar): 

tienes una versión en producción (Blue) que sigue atendiendo a todos los usuarios mientras tú preparas la nueva versión (Green). Durante ese tiempo puedes desplegarla, probarla, verificar que FastAPI funciona, que Docker arranca correctamente, que el modelo carga bien, que las predicciones tienen sentido, etc. Lo importante es que todo eso lo haces sin tocar la versión que están usando los usuarios.

Cuando ya estás convencida de que Green está lista, haces el switch y todo el tráfico pasa de golpe a Green. La ventaja es que no hay ese momento peligroso de "apago una cosa y enciendo otra". Las dos están ya desplegadas y funcionando; simplemente cambias a cuál apuntan los usuarios.

La otra gran ventaja es el rollback. Si cinco minutos después descubres que la nueva versión tiene un problema, no tienes que reconstruir nada ni volver a desplegar. Como Blue sigue existiendo, simplemente vuelves a redirigir el tráfico a Blue y listo.

2. CANARY RELEASE

Aquí no tienes un cambio de golpe como en Blue-Green. Tienes una versión antigua funcionando, por ejemplo el modelo v1, y una versión nueva, el modelo v2. Pero en vez de mandar a todos los usuarios directamente a v2, empiezas con muy poco tráfico.

Por ejemplo:

95% usuarios → modelo v1
5% usuarios  → modelo v2

Ese 5% es el “canario”. Sirve para probar la nueva versión en condiciones reales, pero con riesgo limitado. Si v2 tiene un problema, solo afecta a una pequeña parte del tráfico.

Si todo va bien, subes poco a poco:

80% v1 / 20% v2
50% v1 / 50% v2
20% v1 / 80% v2
100% v2

La gracia es que durante ese proceso monitorizas cosas como latencia, errores, caídas, métricas del modelo y métricas de negocio. En ML mirarías, por ejemplo, si el nuevo modelo tarda más, si devuelve más errores, si cambia demasiado la distribución de scores o si está generando más falsos positivos.

3. SHADOW DEPLOYMENT 

La idea es que tienes:

v1 → Producción LIVE
v2 → Nueva versión

Pero aquí no mandas usuarios a v2 como en Canary. Los usuarios siguen usando 100% la v1.

Lo que haces es copiar las peticiones reales que llegan a v1 y enviarlas también a v2 en segundo plano. Sin embargo, la respuesta de v2 se ignora. El usuario nunca la ve.B Por eso se llama Shadow (sombra): la nueva versión está "siguiendo" a la antigua, viendo exactamente el mismo tráfico real, pero sin afectar a nadie.

La ventaja es enorme porque puedes comprobar:

- Si la API funciona.
- Si la latencia es aceptable.
- Si el modelo responde correctamente.
- Si consume demasiada memoria.
- Si las predicciones son razonables.

Todo ello con tráfico real. Y si v2 explota los usuarios ni se enteran porque siguen usando v1 ✔

> Por eso muchas empresas hacen algo parecido a: Shadow->Canary->Producción completa
> Primero observan cómo se comporta el nuevo modelo sin riesgo alguno. Luego lo enseñan a un pequeño porcentaje de usuarios. Finalmente, si todo va bien, lo despliegan para todos.

> NOTA: En contenedores Docker se utiliza habitualmente 0.0.0.0 para que el servidor sea accesible desde fuera del contenedor. Si se utilizara 127.0.0.1, el servicio únicamente aceptaría conexiones locales dentro del propio contenedor y Docker no podría redirigir correctamente las peticiones externas. Queremos que Uvicorn escuche peticiones que vienen desde fuera del contenedor.

- 127.0.0.1 = escucha solo desde donde estás tú uvicorn.

- 0.0.0.0 = escucha desde cualquier interfaz de red disponible.

>El modelo mental Host → Contenedor → Uvicorn sigue existiendo tanto en local como en cloud. Lo que ocurre es que en plataformas cloud muy gestionadas parte de la configuración de red y de puertos deja de ser responsabilidad del usuario y pasa a estar abstraída por la plataforma.

> Y por eso en el mundo real muchísimos Dockerfile de FastAPI tienen exactamente el mismo CMD tanto si van a ejecutarse:

- En local
- En EC2

> porque Uvicorn sigue estando dentro del contenedor y sigue necesitando escuchar en: 0.0.0.0 para aceptar tráfico que llega desde fuera de él.

## MONITORIZACIÓN

Una vez el modelo queda desplegado, con todas sus particularidades concretas, mas o menos automatizado, se comienza con la monitorización del modelo. EL MODELO YA ESTÁ EN PRODUCCIÓN Y PARECE QUE HEMOS TERMINADO, PERO NO:

- El despliegue de un modelo no garantiza su utilidad a largo
plazo. Una vez en producción, las condiciones cambian: los datos
evolucionan, los contextos de uso se modifican y los patrones que el
modelo aprendió pueden dejar de ser válidos. Si no monitorizamos
lo que está ocurriendo con el modelo en tiempo real, es fácil que su
rendimiento se degrade sin que lo detectemos a tiempo.

### DATA DRIFT (más frecuente-> potencial efecto pero más gradual)

El data drift se refiere a un cambio en la distribución de las variables de entrada. Es decir, los
datos que el modelo recibe en producción dejan de parecerse a los
datos con los que fue entrenado. Esto puede deberse, por ejemplo,
a una nueva fuente de datos, un cambio en la estacionalidad o una
modificación en el comportamiento de los usuarios.

### CONCEPT DRIFT (menos frecuente pero más crítico-> afecta al rendimiento directamente)

El concept drift, en cambio, implica un cambio en la relación entre las variables
de entrada y la variable objetivo. Lo más problemático del concept
drift es que no siempre se detecta revisando solo las distribuciones
de los datos: puede que los inputs no cambien, pero la relación con
el output sí.

Cambia la definición de “fraude” en un sistema de pagos, todo sigue igual, la entrada pero la manera en la que se relaciona con el objetivo que es saber si hay fraude la entrada ya no es la misma porque el fraude se define ahora de manera diferente.

NECESITO ETIQUETAS PARA SABER LO QUE ES EL FRAUDE REAL Y LO QUE ESTÁ PREDICIENDO CON LA ENTRADA. 
> El modelo sigue viendo datos parecidos, pero la realidad que intenta modelar ha cambiado.


Además tenemos que monitorizar con métricas para ver si ocurre alguno de estos problemas mencionados. TENEMOS QUE DIFERENCIAR LAS METRICAS DEL MODELO DE LAS METRICAS DEL SERVICIO

### METRICAS OPERATIVAS  ¿La infraestructura funciona?

- Latencia
- Tiempo de respuesta
- CPU
- Memoria
- Errores HTTP
- Disponibilidad

### METRICAS DEL MODELO 

- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- Log-Loss

> Hay un problema en producción que no teníamos al entrenar el modelo en desarrollo y es que ahora no tengo manera de saber si lo que he respondido a una petición en producción es correcto. Por eso:

### METRICAS PROXY (en ausencia de etiquetas-> buscamos señales de alerta)

No tienes manera de calcular las metricas en ese momento-> a lo mejor en tres meses recibes la respuesta correcta. No obstante puedes ir viendo si hay comportamientos raros en la distribución de las predicciones

Otra estrategia consiste en monitorizar la varianza o la
entropía de las predicciones, buscando comportamientos anómalos
que podrían indicar overfitting local o pérdida de generalización.


### HERRAMIENTAS DE MONITORIZACIÓN GENERALES 

- Prometheus Se dedica a recoger métricas. Por ejemplo:

CPU = 80%
RAM = 4 GB
Latencia = 250 ms

Las va almacenando.

- Grafana Se conecta a Prometheus y las dibuja. Por ejemplo Dashboard con gráficas de:

Latencia
CPU
Memoria
Errores

### HERRAMIENTAS DE MONITORIZACIÓN SON DE ML 

- EvidentlyAI 

- NannyML 

Son específicos de ML y buscan drift, cambios de predicciones(metricas proxy), problemas en el modelo

EvidentlyAI y NannyML permiten monitorizar la salud del modelo y detectar principalmente Data Drift y cambios anómalos en las predicciones. El Concept Drift es más difícil de detectar porque requiere conocer las etiquetas reales para comprobar si la relación entre las variables de entrada y salida ha cambiado.

### CONCEPTOS EXTRA DE LA MONITORIZACIÓN

- Model registry-> versionado de modelos, git de modelos Métricas
Artefactos
Estados (Staging/Production)
Firmas entrada/salida
Metadatos

> MLflow es una plataforma de MLOps que incluye un Model Registry (también hace experiments, metricas, tracking, registry)

Documentación que premite trazabilidad-lineage: 

- Model Card Es la documentación del modelo.

- Data Card Es la documentación de los datos.

- Firmas criptográficas: Garantizan integridad de los artefactos.

## Creo que hay drift por ejemplo con EvidentlyAI vs tengo evidencia estadística de que hay drift

Hasta el momento hablábamos de distribución de predicciones como sintomas de alerta, accuracy bajo-> señales. Veámoslo estadísticamente

1. Kolmogorov-Smirnov (KS Test<0.05 rechazas hipótesis nula y supones que hay alguna varaible que empieza a alejarse)-> ¿Los datos de producción se parecen a los datos de entrenamiento?

Compara los datos de entrada de train y los datos de entrada en producción -> las distribuciones de entrada vs las distribucioens de salida-> solo puede detectar Data Drift porque se enfoca en las entradas. 

> El enfoque es en batches. Cada domingo acumulo los datos de entrada y los comparo -> LO HACE EVIDENTLYAI

2. CUSUM CUMULATIVE SUM ¿hay un cambio sostenido en el comportamiento del sistema? ¿ SE ESTÁN ACUMULANDO DESVIACIONES?

Comparas métricas a lo largo del tiempo. Suma desviaciones pequeñas a lo largo del tiempo-> cumative sum y cuando se supera un umbral salta una alarma YA QUE DETECTA PEQUEÑOS CAMBIOS PERSISTENTES

3. Page-Hinkley ¿Ha cambiado la media del comportamiento del sistema?

Monitorizas error medio

> Kolmogorov-Smirnov compara distribuciones para detectar Data Drift. CUSUM y Page-Hinkley analizan cambios en series temporales de métricas o errores del modelo, siendo especialmente útiles para detectar degradaciones progresivas y posibles casos de Concept Drift (SI SE USA LATENCIA A LO LARGO DEL TIEMPO CLARAMENTE NO ES PARA CONCEPT)

## REENTRENAMIENTO Y AUTOMATIZACIÓN

HEMOS HABLADO DE MONITORIZAR TODO EN PRODUCCIÓN, obviamente, va a haber algun momento en el que se va a detectar drift o degradación del modelo y va a llegar el momento de reentrenar el modelo. Por eso, la monitorización se incluye como paso en MLOps, porque es la gracia para poder pasar al paso de reentrenamiento. 

### REENTRENAMIENTO PROGRAMADO

Una primera aproximación donde se establece una frecuencia
fija para reconstruir el modelo con los datos más recientes. Esta
estrategia es sencilla y fácil de automatizar en pipelines, pero tiene el
inconveniente de no adaptarse a la dinámica real de los datos. Si los
cambios se producen antes de lo previsto, el modelo puede quedar
desfasado durante semanas; si los datos se mantienen estables,
estaremos gastando recursos en un reentrenamiento innecesario.


### REENTRENAMIENTO DISPARADO POR EVENTOS (EJEMPLO DE EVENTOS-> ks test con data drift, cusum detecta degradación, accuracy baja...)

En este caso, el modelo se actualiza solo cuando se detecta una
condición concreta, como una caída significativa en las métricas
de rendimiento o la detección de drift por encima de un umbral.
Esta estrategia es más eficiente, ya que responde a señales reales
en lugar de a un calendario fijo. Sin embargo, requiere un sistema
de monitorización robusto que pueda discriminar entre cambios
relevantes y fluctuaciones normales, para evitar falsas alarmas y
reentrenamientos innecesarios.

- Histéresis

No activas y desactivas exactamente en el mismo punto.

- Cooldown (periodo de enfriamiento)

Reentrené hoy
↓ 
No permito otro reentreno durante 7 días aunque vuelva a saltar una alarma

 
### ENFOQUES HÍBRIDOS

También es posible optar por enfoques híbridos, combinando lo
mejor de ambos mundos. Por ejemplo, podemos programar un
reentrenamiento mensual, pero añadir disparadores adicionales
que activen procesos extraordinarios si se detectan desviaciones
críticas. De esta forma, garantizamos una actualización regular
sin perder capacidad de reacción ante cambios inesperados.

> ¿QUIEN ME REENTRENA TODO? Airflow permite automatizar pipelines de MLOps, ejecutando tareas como extracción de datos, validación, entrenamiento, evaluación y despliegue. Puede lanzar reentrenamientos programados o activarlos por eventos cuando herramientas como EvidentlyAI detectan drift significativo.


- CI: CONTINOUS INTEGRATION

Cada vez que alguien cambia el código se comprueba automáticamente que no haya roto nada (antes de lelgar a producción).

Al hacer un git push, de manera automática hay tests, validaciones, listings para ver si no se ha roto nada. Evidentemente, en caso de que algo falle, no se despliega y se queda allí. 

- CD: CONTINOUS DEPLOYMENT

Si todo ha pasado los tests, automatizamos el despliegue. Te crea una imagen con una imagen de version con tag, luego te la sube a un Registry de los vistos anteriormente y finalmente se despliega la imagen en un servidor (hace docker pull de la imagen)

- CT: CONTINOUS TRAINING (LO PUEDE EJECUTAR AIRFLOW)

Ante la aparición de Data Drift o Concept Drift el modelo puede degradarse. Por ello se puede automatizar el reentrenamiento. Nuevos datos, reentrenas el modelo, evaluas metricas y guardas modelo 


> CASO 1: MODIFICAS API-> CI Y CD
> CASO 2: MODIFICAS MODELO CT->CI-> CD

En un entorno de MLOps, una vez que el modelo ha sido desplegado mediante una API REST contenerizada con Docker y ejecutada en una plataforma cloud, el trabajo no termina. El modelo entra en una fase de monitorización continua donde se supervisan tanto métricas operativas, como latencia o errores HTTP, como métricas relacionadas con el comportamiento del modelo. Herramientas como Prometheus y Grafana permiten monitorizar la infraestructura, mientras que soluciones como EvidentlyAI o NannyML ayudan a detectar posibles problemas de Data Drift o degradación del rendimiento. Cuando se detectan cambios significativos mediante técnicas estadísticas como Kolmogorov-Smirnov o mediante señales obtenidas de métricas proxy, puede activarse un proceso de Continuous Training (CT), encargado de reentrenar automáticamente el modelo con datos más recientes.

Este reentrenamiento suele orquestarse mediante herramientas como Airflow o Kubeflow, que automatizan todo el pipeline: extracción de datos, validación, entrenamiento, evaluación y generación de una nueva versión del modelo. Sin embargo, generar un nuevo modelo no implica desplegarlo directamente. El nuevo artefacto debe pasar por mecanismos de gobernanza, versionado y validación utilizando herramientas como MLflow y los Model Registry. Una vez validado, entra en juego el ciclo de Continuous Integration (CI), donde se ejecutan pruebas automáticas para garantizar que ni el código ni el modelo introducen errores. Si todas las verificaciones son satisfactorias, el proceso de Continuous Deployment (CD) construye una nueva imagen Docker, la publica en un Registry y la despliega inicialmente en un entorno de staging para realizar pruebas finales.

Finalmente, si el rendimiento cumple los umbrales establecidos, la nueva versión se promueve a producción, manteniendo siempre la posibilidad de rollback hacia versiones anteriores en caso de problemas. De esta forma, CI garantiza la calidad del código, CD automatiza el despliegue y CT asegura que el modelo continúe adaptándose a los cambios del negocio y de los datos. La combinación de estas tres prácticas constituye la base de una estrategia moderna de MLOps, permitiendo mantener modelos fiables, trazables y alineados con la realidad operativa a lo largo del tiempo.